In [1]:
import os
from openai import OpenAI
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

In [38]:
import pypdf

pdf_path = (r"D:\Ai engineering course\Day_25\Micro_Syallabus.pdf")

doc = fitz.open(pdf_path)
text = ""
for page in doc :
    text += page.get_text() + "\n"


chunk_size = 1000
overlap = 30

chunks = []

start = 0
while start<len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

# for i, chunk in enumerate(chunks):
#     print(f"\n------Chunk {i}------")
#     print(chunk)
    


In [43]:
import chromadb
import json
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# STEP 1: THE DATA
# ==========================================


# ==========================================
# STEP 1.5: AUTO-METADATA (GEMINI)
# ==========================================
llm = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),  # key read from .env, never hardcoded
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

TOPICS = ["RAG", "AI_ML", "Course", "Academy"]

def generate_metadata(chunks):
    numbered = "\n\n".join(f"[{i}] {c}" for i, c in enumerate(chunks))
    resp = llm.chat.completions.create(
        model="gemini-3.6-flash",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content":
            f'Label each chunk. topic must be one of {TOPICS}.\n'
            f'Return {{"items":[{{"topic":..., "summary":"one sentence"}}]}} '
            f'— one per chunk, same order, JSON only.\n\n{numbered}'}]
    )
    items = json.loads(resp.choices[0].message.content)["items"]
    return [{"topic": it.get("topic") if it.get("topic") in TOPICS else "unknown",
             "summary": it.get("summary", "")[:300]}
            for it in items]

metadatas = generate_metadata(chunks)

# Check what Gemini actually labeled before you filter on it
for c, m in zip(chunks, metadatas):
    print(f"{m['topic']:<10} | {c[:50]}")

# ==========================================
# STEP 2: THE EMBEDDER (TRANSLATOR)
# ==========================================
vectorizer = TfidfVectorizer()
chunk_embeddings = vectorizer.fit_transform(chunks).toarray()

def embed(text):
    """Helper function to translate a single question into math"""
    return vectorizer.transform([text]).toarray()[0]

# # ==========================================
# # STEP 3: SETTING UP THE DATABASE
# # ==========================================
# client = chromadb.Client()
# col = client.create_collection(
#     name="notes_4",
#     metadata={"hnsw:space": "cosine"}
# )

# ==========================================
# STEP 4: INGESTION
# ==========================================
# ids = [str(i) for i in range(len(chunks))]
# col.add(
#     ids=ids,
#     embeddings=chunk_embeddings.tolist(),
#     documents=chunks,
#     metadatas=metadatas
# )
client = chromadb.Client()
col = client.get_collection("notes_4")

# ==========================================
# STEP 5: THE SEARCH (WITH A FILTER!)
# ==========================================
question = "How do i become a ai ml engineer?"
results = col.query(
    query_embeddings=[embed(question).tolist()],
    n_results=2,
    where={"topic": "Academy"}
)

# print("--- SEARCH RESULTS ---")
# for doc in results['documents'][0]:
#     print(f"- {doc}")
# print("ran")

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 16.36771082s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 'q

list

In [26]:
chunks

['SAARATHI ACADEMY\nfor Digital Excellence\nAI Engineering & Machine Learning\nCOURSE BRIEF\nSKILLS & TOOLS YOU WILL MASTER\nPython + scikit-learn\nPyTorch + Transformers\nLangChain + LangGraph\nPC RAG + Pinecone\nAM Agents + Mem0 + MCP\nFastAPI + Docker\n10 Reasons This Works.\nWhy students choose Saarathi - and how each card helps you ship a real career.\nsaarathiacademy.com.np/ai-engineering-and-machine-learning-\ncourse-in-nepal\n+977-9744442469 \xa0·\xa0 +977-\n9761095364\nTR-10\n01\n01\n02\n02\n03\n03\n04\n04\n05\n05\n06\n06\n07\n07\n08\n08\n09\n09\n10\n10\n10 Max / Batch\nNot 50. Not 30. Ten. Every\nstudent seen, heard, and\nunblocked - every class.\nSunday\nOpen\nClassroom\nEvery Sunday the\nclassroom \nis\nopen. Come in,\npair-program,\nget unstuck, or\njust focus.\nDay-by-Day\nSyllabus\nFull \nsyllabus.\nEvery day: what\nto read, what to\nbuild, what the\noutcome is.\nRevision Every 5th\nDay\nEvery fifth class is a revision\nsession. Nothing piles up or\nslips through.\nWeek 

In [ ]:
# client = chromadb.Client()

# # TF-IDF dimension changes whenever chunks change — rebuild, never reuse
# try:
#     client.delete_collection("notes_4")
# except Exception:
#     pass


# )